# Portfolio analysis from the engine

This notebook drives `quant/` directly: the same code the API calls, with no web
server and no database involved. That is the point of keeping the engine free of
framework imports — a result reproduced here is the result the app shows.

Install the engine first (`pip install -e ".[dev]"`) and, for the charts,
`pip install matplotlib`.

In [ ]:
import sys
from pathlib import Path

# The engine is a plain package at the repository root: no server, no database.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("ready")

## 1. Prices

`DATA_SOURCE` decides where the app gets prices; here we call the provider
directly. `synthetic` needs no network and is deterministic; swap in `"yahoo"`
for real adjusted closes (needs `pip install yfinance`).

In [ ]:
from quant.data.providers import fetch_prices
from quant.data.validation import to_price_matrix, validate_prices

SOURCE = "synthetic"          # or "yahoo"
TICKERS = ["AAPL", "MSFT", "NVDA", "JPM", "XOM", "SPY", "TLT", "GLD"]

raw = fetch_prices(SOURCE, TICKERS, "2019-01-02", None)
clean, report = validate_prices(raw)
print(report.as_dict())

prices = to_price_matrix(clean)[TICKERS]
prices.tail()

Validation drops missing days rather than forward-filling them. A filled price
invents a flat day, which pulls measured volatility down and makes the portfolio
look safer than it is.

In [ ]:
(prices / prices.iloc[0]).plot(title=f"Growth of 1 ({SOURCE} data)")
plt.ylabel("multiple of starting value")
plt.show()

## 2. Returns and per-asset statistics

In [ ]:
from quant.features.returns import asset_statistics, simple_returns

rets = simple_returns(prices)
print(f"{len(rets)} daily observations, {rets.index[0].date()} to {rets.index[-1].date()}")

stats = asset_statistics(rets, risk_free_rate=0.04)
stats[["annual_return", "annual_volatility", "sharpe", "cagr", "max_drawdown", "skew", "kurtosis"]]

Negative skew and excess kurtosis are the normal state of equity returns, and
they are why a normal-distribution VaR understates the tail.

## 3. Risk of a portfolio

Weights are given directly here. In the app they come from your holdings valued
at current prices.

In [ ]:
from quant.optimization.estimators import estimate_covariance, estimate_expected_returns
from quant.risk.metrics import concentration, risk_contribution, risk_report

w = pd.Series({"AAPL": 0.18, "MSFT": 0.18, "NVDA": 0.12, "JPM": 0.12,
               "XOM": 0.10, "SPY": 0.15, "TLT": 0.10, "GLD": 0.05})
w = w.reindex(TICKERS).fillna(0.0)

mu = estimate_expected_returns(rets, method="historical")
cov = estimate_covariance(rets, method="ledoit_wolf")

rep = risk_report(rets, w, mu, cov, risk_free_rate=0.04, benchmark=rets["SPY"])
pd.Series({k: v for k, v in vars(rep).items() if isinstance(v, (int, float))})

### Where the risk actually sits

A 10% weight is not 10% of the risk. The decomposition is exact —
`RC_i = w_i (Sigma w)_i / sigma_p` — and the contributions sum to portfolio
volatility.

In [ ]:
rc, rc_pct = risk_contribution(w.to_numpy(), cov.loc[TICKERS, TICKERS].to_numpy())
table = pd.DataFrame({"weight": w, "risk_contribution": rc, "risk_share": rc_pct}, index=TICKERS)

print(f"sum of contributions = {rc.sum():.6f}   portfolio volatility = {rep.volatility:.6f}")
table.sort_values("risk_share", ascending=False)

In [ ]:
ax = table[["weight", "risk_share"]].sort_values("risk_share").plot.barh(
    title="Weight share vs risk share")
ax.set_xlabel("share of portfolio")
plt.tight_layout()
plt.show()

print(concentration(w.to_numpy()))

Anything sitting well to the right of its weight carries more risk than its size
suggests: either it is more volatile than the rest, or it is more correlated
with everything else held.

## 4. Correlation

In [ ]:
from quant.optimization.estimators import correlation_from_covariance

corr = correlation_from_covariance(cov).loc[TICKERS, TICKERS]
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(TICKERS)), TICKERS, rotation=45, ha="right")
ax.set_yticks(range(len(TICKERS)), TICKERS)
for i in range(len(TICKERS)):
    for j in range(len(TICKERS)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, shrink=0.8)
ax.set_title("Correlation: diverging scale, neutral midpoint at 0")
ax.grid(False)
plt.tight_layout()
plt.show()

## 5. Optimization and the efficient frontier

Each frontier point minimizes `w'Sigma w` for a fixed expected return under the
same constraints, so every point is investable under those rules rather than a
point on an unconstrained textbook curve.

In [ ]:
from quant.optimization.optimizer import Constraints, efficient_frontier, optimize

cons = Constraints(max_weight=0.30, long_only=True)

for objective in ("min_variance", "max_sharpe", "risk_parity"):
    res = optimize(mu, cov, objective, cons, risk_free_rate=0.04, sectors={})
    top = sorted(res.weights.items(), key=lambda kv: -kv[1])[:4]
    print(f"{objective:14} vol={res.volatility:6.2%}  ret={res.expected_return:6.2%}  "
          f"sharpe={res.sharpe:5.2f}  top={', '.join(f'{t} {v:.0%}' for t, v in top)}")

In [ ]:
ef = efficient_frontier(mu, cov, cons, risk_free_rate=0.04, sectors={}, n_points=40)
front = pd.DataFrame(ef["frontier"])
rnd = pd.DataFrame(ef["random_portfolios"])
assets = pd.DataFrame(ef["assets"])

fig, ax = plt.subplots()
ax.scatter(rnd["volatility"], rnd["expected_return"], s=6, alpha=0.25, label="random feasible")
ax.plot(front["volatility"], front["expected_return"], lw=2, label="efficient frontier")
ax.scatter(assets["volatility"], assets["expected_return"], s=40, label="individual assets")
for _, a in assets.iterrows():
    ax.annotate(a["ticker"], (a["volatility"], a["expected_return"]),
                textcoords="offset points", xytext=(5, 3), fontsize=8)
tan = ef["tangency"]
ax.scatter([tan["volatility"]], [tan["expected_return"]], marker="*", s=220, label="tangency")
ax.scatter([rep.volatility], [float(w @ mu[TICKERS])], marker="X", s=120, label="this portfolio")
ax.set_xlabel("annualized volatility")
ax.set_ylabel("expected annual return")
ax.set_title("Risk vs expected return: 30% cap, long only")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

The portfolio marker sitting below the curve means it is inefficient **under
this model**: a feasible portfolio offers more expected return at the same risk.
The curve moves when the estimation window or the estimator changes, so treat
its shape as the lesson and its exact position as provisional.

### How much the estimator matters

Worth running before trusting any single optimization.

In [ ]:
rows = []
for ret_method in ("historical", "bayes_stein", "capm"):
    for cov_method in ("sample", "ledoit_wolf", "ewma"):
        m = estimate_expected_returns(rets, method=ret_method, benchmark=rets["SPY"], risk_free_rate=0.04)
        c = estimate_covariance(rets, method=cov_method)
        r = optimize(m, c, "max_sharpe", cons, risk_free_rate=0.04, sectors={})
        rows.append({"returns": ret_method, "covariance": cov_method,
                     "vol": r.volatility, "exp_return": r.expected_return, "sharpe": r.sharpe,
                     "largest_position": float(r.weights.max())})

pd.DataFrame(rows).sort_values("sharpe", ascending=False)

The spread across those nine rows is estimation error, not insight. It is why
the app shows the window and both estimators next to every number, and why
shrinkage is worth preferring over a raw sample mean.